In [ ]:
import pandas as pd
import numpy as np
import netCDF4 as nc
import os
from datetime import timedelta
import glob

In [ ]:
filepath = '/proj6/users/ma/data/output/h8_mcs/Final_MCS_List_genesis.csv'
save_path = '/proj6/users/ma/data/output/h8_lst'
h8_path = '/proj4/data/Himawari8/'
if not os.path.exists(save_path):
    os.makedirs(save_path)
# search_pattern = os.path.join(h8_path, '*', '*.nc')
# file_list = glob.glob(search_pattern)
# file_list.sort()

#file_list = file_list[:24]

In [ ]:
# def dynamic_threshold(lat, lon, hour):
#     is_time = 8 <= hour <= 17
#     base_k = 273.15
    
#     if lat<35 and lon<100:
#         if is_time:
#             return base_k + 5
#         else:
#             return base_k
        
#     elif lat>=40 and lon <= 115:
#         if is_time:
#             return base_k +10
#         else:
#             return base_k 
#     else:
#         if is_time:
#             return base_k + 15
#         else:
#             return base_k + 10

In [ ]:
df = pd.read_csv(filepath)
df['time'] = pd.to_datetime(df['time'])

NC_VAR_NAME = 'tbb_13'
NC_VAR_REF = 'albedo_03'
NC_VAR_SAZ = 'SAZ'  

ALBEDO_THRESH = 0.3
TBB_DAY_THRESH = 275
TBB_Night_THRESH = 273
SAZ_THRESH = 85

extracted_data = []

r = 2 # mean-r
r_var = 8 #vari-r

for idx, row in df.iterrows():
    genesis_time = row['time'] # 这是 MCS 生成时间
    lat = row['center_lat']
    lon = row['center_lon']

    record = {
        'global_id': row['global_id'],
        'genesis_time': genesis_time,
        'lat': lat,
        'lon': lon
    }

    for h in range(1, 7):
        target_time = genesis_time - timedelta(hours=h)# 目标时刻
        col_name = f'TBB_T-{h}h'
        col_var  = f'Var_T-{h}h'
        record[col_name] = np.nan
        record[col_var]  = np.nan

        fname = target_time.strftime('*%Y%m%d_%H%M*.nc') 
        yearmonth = target_time.strftime('%Y%m')
        fpath = os.path.join(h8_path, yearmonth, fname)
        found_files = glob.glob(fpath)

        #print(f"文件数: {len(found_files)}")

        if not found_files:
        #     file_not_found += 1
        #     #print(f"T-{h}h: 未找到文件 {fname}")
             continue

        fpath = found_files[0]
        
        try:
            with nc.Dataset(fpath, 'r') as ds:
                if NC_VAR_NAME not in ds.variables:
                    continue

                lats = ds.variables['latitude'][:]
                lons = ds.variables['longitude'][:]

                if lats.ndim == 1:  # 1维情况
                    lat_idx = (np.abs(lats - lat)).argmin()
                    lon_idx = (np.abs(lons - lon)).argmin()
                    
                elif lats.ndim == 2:  # 2维情况
                    distances = np.sqrt((lats - lat)**2 + (lons - lon)**2)
                    lat_idx, lon_idx = np.unravel_index(
                        distances.argmin(), 
                        distances.shape
                    )

                if lats.ndim == 2:
                    nrows, ncols = lats.shape
                else:
                    nrows, ncols = len(lats), len(lons)

                if (lat_idx - r_var < 0 or lon_idx - r_var < 0 or 
                    lat_idx + r_var + 1 > nrows or 
                    lon_idx + r_var + 1 > ncols):
                    continue

                saz = ds.variables[NC_VAR_SAZ][lat_idx, lon_idx]    
                if np.ma.is_masked(saz):
                    continue

                saz = float(saz)
                is_day = saz < SAZ_THRESH
                
                if NC_VAR_NAME in ds.variables:
                    tbb_block = ds.variables[NC_VAR_NAME][lat_idx-r : lat_idx+r+1, lon_idx-r : lon_idx+r+1]
                    tbb_var_block = ds.variables[NC_VAR_NAME][lat_idx-r_var : lat_idx+r_var+1, lon_idx-r_var : lon_idx+r_var+1]
                
                tbb_block = np.ma.masked_invalid(tbb_block.astype(float))
                tbb_var_block = np.ma.masked_invalid(tbb_var_block.astype(float))

                if is_day:
                    if NC_VAR_REF not in ds.variables:
                        continue
                
                    ref_block = ds.variables[NC_VAR_REF][
                            lat_idx-r:lat_idx+r+1, 
                            lon_idx-r:lon_idx+r+1
                        ]

                    ref_var = ds.variables[NC_VAR_REF][
                        lat_idx-r_var:lat_idx+r_var+1, 
                        lon_idx-r_var:lon_idx+r_var+1
                    ]

                    if np.ma.is_masked(ref_block):
                        ref_block = ref_block.filled(np.nan)
                    if np.ma.is_masked(ref_var):
                        ref_var = ref_var.filled(np.nan)

                    cloud_mask = (ref_block > ALBEDO_THRESH) | (tbb_block < TBB_DAY_THRESH)
                    cloud_mask_ref = (ref_var > ALBEDO_THRESH) | (tbb_var_block < TBB_DAY_THRESH)
                    tbb_block[cloud_mask] = np.nan
                    tbb_var_block[cloud_mask_ref] = np.nan

                    mean_val = np.nanmean(tbb_block)
                    if not np.isnan(mean_val):
                        record[col_name] = mean_val - 273.15

                    valid_count_var = np.count_nonzero(~np.isnan(tbb_var_block))
                    total_count_var = tbb_var_block.size
                    if valid_count_var > (total_count_var * 0.2):
                        var_val = np.nanvar(tbb_var_block)
                        record[col_var] = var_val
                    else:
                        record[col_var] = np.nan
                
                else:                    
                    # 应用TBB阈值(去除高亮温=薄云/晴空)
                    tbb_block[tbb_block < TBB_Night_THRESH] = np.nan
                    tbb_var_block[tbb_var_block < TBB_Night_THRESH] = np.nan
                    
                    # 计算均值
                    mean_val = np.nanmean(tbb_block)
                    if not np.isnan(mean_val):
                        record[col_name] = mean_val - 273.15
                    
                    # 计算方差
                    valid_count_var = np.count_nonzero(~np.isnan(tbb_var_block))
                    total_count_var = tbb_var_block.size
                    if valid_count_var > (total_count_var * 0.2):
                        var_val = np.nanvar(tbb_var_block)
                        record[col_var] = var_val
                    else:
                        record[col_var] = np.nan
                
        except Exception as e:
            pass
        
    extracted_data.append(record)
    if (idx + 1) % 100 == 0:
        print(f"已处理: {idx + 1}/{len(df)}")
    
res_df = pd.DataFrame(extracted_data)
output_filename = 'mcs_h8_lst_var40.csv'
full_save_path = os.path.join(save_path, output_filename)
res_df.to_csv(full_save_path, index=False)
print('end')